# Data Preparation

It prepares/checks the two datasets separately:

- IoT data: `Normal` and `Cooking`
- Blattmann data: `Normal` and `Fire`

Class mapping:

```text
0 = Normal
1 = Cooking
2 = Fire
```


#### 1. Imports


In [37]:
import re
import json
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


#### 2. Settings


In [39]:
# Define paths, feature columns, and class names.
notebook_folder = Path('.').resolve()
parent_folder = notebook_folder.parent

iot_raw_folder = parent_folder / 'Iot team data collected'
iot_file = notebook_folder / 'iot_clean_data.xlsx'
blattmann_file = notebook_folder / 'blattmann_clean_data.xlsx'

feature_col = ['temperature', 'humidity', 'tvoc_ppb', 'eco2_ppm']
class_names = ['Normal', 'Cooking', 'Fire']


#### 3. IoT Scenario Mapping

This maps each raw IoT scenario folder to a class.

If the raw IoT folder is not available, we load `iot_clean_data.xlsx` instead.


In [40]:
# Map each IoT scenario folder to a class.
scenario_map = {
    'No smoke Data': ('Normal', 0, 'message.txt'),
    'Cooking smoke data (5 minutes before event)': ('Normal', 0, 'message (1).txt'),
    'Cooking smoke data (during event)': ('Cooking', 1, 'message (2).txt'),
    'Cooking smoke data (right after event has happened, 5 minutes)': ('Cooking', 1, 'message (3).txt'),
    'Steam fog (data for preventing false alarms, 5 minutes before event)': ('Normal', 0, 'message (4).txt'),
    'Steam fog (during steaming)': ('Cooking', 1, 'message (5).txt'),
}


#### 4. Load IoT Data

- build IoT data from raw message files if they exist
- otherwise use the already-clean IoT Excel file


In [41]:
# Load IoT data from raw files if possible; otherwise load the clean Excel file.
json_pattern = re.compile(r'\{[^{}]+\}')
rows = []

if iot_raw_folder.exists():
    for folder, (label, class_id, file_name) in scenario_map.items():
        file_path = iot_raw_folder / folder / file_name
        if not file_path.exists():
            print('Missing:', file_path)
            continue

        for json_text in json_pattern.findall(file_path.read_text(errors='ignore')):
            try:
                data = json.loads(json_text)
                rows.append({
                    'label': label,
                    'class_id': class_id,
                    'temperature': data.get('temperature'),
                    'humidity': data.get('humidity'),
                    'tvoc_ppb': data.get('tvoc'),
                    'eco2_ppm': data.get('eco2'),
                })
            except json.JSONDecodeError:
                pass

    iot_data = pd.DataFrame(rows).dropna(subset=feature_col)
    iot_data.to_excel(notebook_folder / 'iot_clean_data.xlsx', index=False)
    print('IoT data created from raw message files.')
else:
    iot_data = pd.read_excel(iot_file)
    print('Raw IoT folder not found. Loaded iot_clean_data.xlsx.')

print('IoT samples:', len(iot_data))
print(iot_data['label'].value_counts().to_string())
display(iot_data.head())


Raw IoT folder not found. Loaded iot_clean_data.xlsx.
IoT samples: 549
label
Cooking    329
Normal     220


,label,class_id,temperature,humidity,tvoc_ppb,eco2_ppm
0,Normal,0,21.8,32,29,413
1,Normal,0,21.8,32,35,426
2,Normal,0,21.8,32,28,409
3,Normal,0,21.8,32,25,402
4,Normal,0,21.8,32,29,413


#### 5. IoT Quality Checks

We check null values and duplicate rows in the IoT data.


In [31]:
# Check IoT null values and duplicates.
print('IoT null values:')
display(iot_data.isnull().sum().to_frame('null_count'))

print('IoT duplicate rows:', iot_data.duplicated().sum())


IoT null values:


,null_count
label,0
class_id,0
temperature,0
humidity,0
tvoc_ppb,0
eco2_ppm,0


IoT duplicate rows: 117


#### 6. Load and Remap Blattmann Data

The idea is:

```text
No Smoke / Normal -> class_id 0 -> Normal
Smoke / Fire      -> class_id 2 -> Fire
```


In [44]:
# Load and remap Blattmann data to the 3-class project format.
blattmann_data = pd.read_excel(blattmann_file, sheet_name='Blattmann Processed')

if 'label' in blattmann_data.columns:
    blattmann_data = blattmann_data.rename(columns={'label': 'blattmann_label'})

blattmann_data['class_id'] = blattmann_data['blattmann_label'].map({
    'No Smoke': 0,
    'Smoke': 2,
    'Normal': 0,
    'Fire': 2,
    0: 0,
    2: 2,
})

blattmann_data['label'] = blattmann_data['class_id'].map({
    0: 'Normal',
    2: 'Fire',
})

blattmann_data = blattmann_data.dropna(subset=['class_id'] + feature_col)
blattmann_data['class_id'] = blattmann_data['class_id'].astype(int)

print('Blattmann samples:', len(blattmann_data))
print(blattmann_data['label'].value_counts().to_string())
display(blattmann_data.head())


Blattmann samples: 62630
label
Fire      44757
Normal    17873


,temperature,humidity,tvoc_ppb,eco2_ppm,blattmann_label,class_id,label
0,20.000,57.36,0,400,No Smoke,0,Normal
1,20.015,56.67,0,400,No Smoke,0,Normal
2,20.029,55.96,0,400,No Smoke,0,Normal
3,20.044,55.28,0,400,No Smoke,0,Normal
4,20.059,54.69,0,400,No Smoke,0,Normal


#### 7. Blattmann Quality Checks

We check null values and duplicate rows in the Blattmann data.


In [45]:
# Check Blattmann null values and duplicates.
print('Blattmann null values:')
display(blattmann_data.isnull().sum().to_frame('null_count'))

print('Blattmann duplicate rows:', blattmann_data.duplicated().sum())


Blattmann null values:


,null_count
temperature,0
humidity,0
tvoc_ppb,0
eco2_ppm,0
blattmann_label,0
class_id,0
label,0


Blattmann duplicate rows: 41


#### 8. Final Data Preparation Notes

Data preparation is now separate from model training.

The Random Forest notebook should load the two datasets separately:

```text
iot_clean_data.xlsx
blattmann_clean_data.xlsx
```
